# H2b · Graded test of the SQL agent (O2 / O3 / O4)

**What it decides:** which model, if any, turns free-form questions into SQL over H2's per-building table in the deployed system, without adding a model the system doesn't need.

**Why.** H2 compared two LLMs on 100 counting questions. Every one of them needs the same query (`SELECT COUNT(*) ... WHERE image = ? AND class_name = ?`), so that comparison can't separate the models, and in the deployed system counting needs no model at all: a fixed query does it. A second model also costs memory on low-resource hardware. H2b grades the agent on the open questions a responder would ask, and prices each candidate in memory and latency.

**Design** (each choice follows published practice; references at the end):
1. **A question bank fixed before any run (Cell 4).**
   - 29 answerable questions in 12 groups: counts, totals, percentages, areas, size thresholds, distances, "no destroyed building nearby" (a NULL trap), positions, rankings, yes/no, pairwise distances, combined conditions.
   - Each in 3 phrasings (plain, responder words, indirect): rewording breaks text-to-SQL models, and in Dr.Spider even the most robust model lost 14%.
   - Each on 3 images: 2 typical, and 1 where the answer is zero or none.
   - 12 questions the tables can't answer (injuries, roads, cause, height, cost, square metres...), 3 phrasings each.
2. **Test-suite accuracy (Zhong, Yu & Klein 2020).** A query can match the right answer on one table by chance, for example when it ignores NULLs, flips the y axis, or double-counts pairs that happen to be absent. So every answer runs on 5 versions of the table: the real one, classes redrawn, sizes and positions redrawn, no destroyed buildings, and no buildings. It's correct only if it matches the reference query on all 5. Cell 7 checks that the suite catches 6 such near-miss queries and accepts 7 correct rewrites.
3. **No ambiguous instances.** An image isn't used for a question when the answer depends on a tie (two equally large buildings) or on a building exactly at a threshold or midline.
4. **Refusals and a reliability score (TrustSQL; the EHRSQL 2024 shared task, whose main metric is RS(10)).**
   - +1 for a right answer, or for refusing an unanswerable question;
   - 0 for refusing an answerable one;
   - −10 for a wrong answer, or for answering an unanswerable question. A wrong number is costly in disaster response.

   A query that still fails after the retries counts as a refusal: the system shows an error, not a number.
5. **One prompt for every model, fixed before the run (Cell 8).** It shows the tables as CREATE TABLE statements with column comments (the code representation of DAIL-SQL and Rajkumar et al.), gives 4 worked examples including one refusal, and states the rules. Results are split into question types an example shows ("seen") and those it doesn't ("unseen").
6. **Retries with the error message** (up to 2), as in Self-Debugging; first-try accuracy is reported too.
7. **Statistics:** 95% intervals by a cluster bootstrap over questions, because the 9 items of one question aren't independent; differences between models by a paired bootstrap on the same draws.

**Candidates**, each in 4-bit NF4 (the low-resource form) and in fp16 (to show what 4-bit costs):

| Model | Role | Extra memory in the system |
|---|---|---|
| our 7B, `AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP`, text only | the VLM the system ships now | none |
| `Qwen/Qwen2.5-VL-3B-Instruct` | the base of the distilled 3B (L2's student), its stand-in until L2 runs | none once the 3B ships |
| `Qwen/Qwen2.5-Coder-1.5B-Instruct` | a separate SQL model, the smallest | its own weights |
| `Qwen/Qwen2.5-Coder-3B-Instruct` | a separate SQL model, H2's agent | its own weights |

**Decision rule, fixed 22 Oct before any run.**
- A configuration passes if its answerable test-suite accuracy is at least 90% and its RS(10) is above 0.
- Benchmark counting questions always use the fixed query, never a model.
- If our VLM passes in 4-bit, it writes the SQL, with no second model. The base VL-3B stands in for the distilled 3B; re-run H2b on the distilled 3B before deciding the final system.
- If only a separate model passes, the smallest passing one is the fallback, with its extra memory stated, and the team decides whether free-form questions are worth it.
- If nothing passes, the system ships fixed queries only.

**How to run on Kaggle**
1. GPU T4 ×2, internet ON.
2. Input: H2's output (its notebook output, or a dataset made from it). This notebook reads `buildings.sqlite`.
3. Secret `HF_TOKEN` (our 7B's repo).
4. `SMOKE_TEST = True` first: 4 questions and 2 unanswerable ones, one phrasing and one image each, with the 1.5B coder in 4-bit; about 10 min. Then Save & Run All: about 1–1.5 h. No new configuration starts after `TIME_BUDGET_H` (3 h).
5. Outputs in `/kaggle/working/H2b/`: `summary.csv`, `by_group.csv`, `by_phrasing.csv`, `by_unanswerable.csv`, `pairwise.csv`, `graded.csv`, `items.csv`, `gold.json`, `answers.jsonl`, `runs.jsonl`, `prompt.txt`, `decision.md`, `config.json`, `README.md`.

**References**
- Zhong, Yu & Klein. Semantic Evaluation for Text-to-SQL with Distilled Test Suites. EMNLP 2020.
- Lee et al. TrustSQL: Benchmarking Text-to-SQL Reliability with Penalty-Based Scoring. 2024. Lee et al. Overview of the EHRSQL 2024 Shared Task on Reliable Text-to-SQL Modeling on Electronic Health Records. ClinicalNLP 2024.
- Chang et al. Dr.Spider: A Diagnostic Evaluation Benchmark towards Text-to-SQL Robustness. ICLR 2023.
- Gao et al. Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation (DAIL-SQL). VLDB 2024.
- Rajkumar, Li & Bahdanau. Evaluating the Text-to-SQL Capabilities of Large Language Models. 2022.
- Chen et al. Teaching Large Language Models to Self-Debug. ICLR 2024.
- Hui et al. Qwen2.5-Coder Technical Report. 2024.

In [ ]:
# ── Cell 1: Environment: Kaggle's torch and transformers, plus bitsandbytes for 4-bit (as in L0 step 2) ──
import importlib.metadata as _md
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 20 Oct)"

import torch
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "bitsandbytes", "accelerate")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

In [ ]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
import gc
import glob
import json
import math
import os
import re
import shutil
import sqlite3
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

OUT_DIR = Path("/kaggle/working/H2b")
SMOKE_TEST = False                 # True = 4 answerable and 2 unanswerable questions, one phrasing, one image, the 1.5B coder in 4-bit
SEED = 42
POOL_IMAGES = 600                  # images searched for each question's instances (the reference queries are cheap)
IMAGES_PER_QUESTION = 3            # 2 typical images + 1 whose answer is zero or none
MODELS = [  # (short name, Hugging Face id, "vl" = a Qwen2.5-VL used as text only, role in the system)
    ("our-7B", "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP", "vl", "the VLM the system ships now: no extra memory"),
    ("VL-3B", "Qwen/Qwen2.5-VL-3B-Instruct", "vl", "base of the distilled 3B (L2), its stand-in: no extra memory"),
    ("Coder-1.5B", "Qwen/Qwen2.5-Coder-1.5B-Instruct", "text", "a separate SQL model, the smallest"),
    ("Coder-3B", "Qwen/Qwen2.5-Coder-3B-Instruct", "text", "a separate SQL model, H2's agent"),
]
PRECISIONS = ["nf4", "fp16"]       # nf4 = 4-bit, the low-resource form; fp16 shows what 4-bit costs
BATCH_SIZE = 16                    # batched greedy decoding, checked against one-at-a-time decoding (LATENCY_ITEMS)
LATENCY_ITEMS = 24                 # per configuration: one-at-a-time latency and peak memory, and the batching check
AGREE_MIN = 0.90                   # below this agreement a configuration is decoded one at a time instead
MAX_NEW_TOKENS = 256
MAX_SQL_RETRIES = 2                # extra attempts after a failed query, each with the error message
SQL_TIMEOUT_S = 3
TIME_BUDGET_H = 3.0                # no new configuration starts after this; the report is still written
PASS_TS_ACC = 90.0                 # decision rule, fixed 22 Oct before any run: answerable test-suite accuracy (%)
RS_PENALTY = 10                    # ... and RS(10) above 0 (TrustSQL, EHRSQL 2024: -10 for each wrong answer)
BOOTSTRAP = 2000
HF_CACHE_CLEANUP = True            # delete each model's download after its last configuration (disk space)

if SMOKE_TEST:
    MODELS, PRECISIONS = [m for m in MODELS if m[0] == "Coder-1.5B"], ["nf4"]
    POOL_IMAGES, IMAGES_PER_QUESTION, LATENCY_ITEMS, BOOTSTRAP = 200, 1, 2, 200

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

H2_DB = find_one("buildings.sqlite", "attach H2's output (the h2-sql-counting-agent notebook output)")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS = [(name, hf_id, kind, role, prec) for name, hf_id, kind, role in MODELS for prec in PRECISIONS]
print(f"✓ H2b{'  [SMOKE TEST]' if SMOKE_TEST else ''}: table {H2_DB}")
print(f"  {len(CONFIGS)} configurations: {[f'{c[0]} {c[4]}' for c in CONFIGS]}")
print(f"  GPUs: {torch.cuda.device_count()}")

In [ ]:
# ── Cell 3: H2's table, and its schema as the model sees it ─────────────────
# The column notes are H2's, written for the model. The prompt shows the tables as CREATE TABLE statements with
# the notes as comments (the code representation of DAIL-SQL and Rajkumar et al.).
COLUMN_NOTES = {
    "image": "after-image file name",
    "building_id": "building number within that image",
    "class_id": "1 = intact, 2 = damaged, 3 = destroyed",
    "class_name": "'intact', 'damaged' or 'destroyed'",
    "area_px": "building area in pixels",
    "centroid_x": "building centre, pixels from the left edge",
    "centroid_y": "building centre, pixels from the top edge (y grows downwards)",
    "bbox_x": "bounding box left",
    "bbox_y": "bounding box top",
    "bbox_w": "bounding box width",
    "bbox_h": "bounding box height",
    "dist_to_nearest_destroyed_px": "centre-to-centre distance in pixels to the nearest OTHER destroyed building "
                                    "in the same image; NULL if there is none",
}
CLASS_BY_ID = {1: "intact", 2: "damaged", 3: "destroyed"}

src = sqlite3.connect(f"file:{H2_DB}?mode=ro", uri=True)
COLS = [(r[1], r[2]) for r in src.execute("PRAGMA table_info(buildings)")]
assert [c for c, _ in COLS] == list(COLUMN_NOTES), f"H2's buildings table has the columns {COLS}"
assert [r[1] for r in src.execute("PRAGMA table_info(images)")] == ["image", "width", "height"]
BUILDINGS = pd.read_sql("SELECT * FROM buildings", src)
IMAGES = pd.read_sql("SELECT * FROM images", src)
src.close()
SIZE = {r.image: (int(r.width), int(r.height)) for r in IMAGES.itertuples()}

SCHEMA_SQL = ("CREATE TABLE buildings (  -- one row per building: an 8-connected blob of one class in the damage map\n"
              + "\n".join(f"  {c} {t},  -- {COLUMN_NOTES[c]}" for c, t in COLS)
              + "\n  PRIMARY KEY (image, building_id)\n);\n"
              "CREATE TABLE images (  -- one row per image\n"
              "  image TEXT PRIMARY KEY,\n  width INTEGER,  -- pixels\n  height INTEGER  -- pixels\n);")
print(f"✓ {len(BUILDINGS):,} buildings in {len(IMAGES):,} images {dict(BUILDINGS['class_name'].value_counts())}")
print(SCHEMA_SQL)

In [ ]:
# ── Cell 4: The question bank, fixed 22 Oct before any run ──────────────────
# 29 answerable questions in 12 groups, each in 3 phrasings (plain, responder words, indirect), and 12 that the
# tables cannot answer. Every answerable question has a reference ("gold") query; :image is the instance's image.
#   kind: how an answer is read (number, bool, class, list); tol: the allowed difference for numbers
#   zero_if_none: no rows or NULL means 0 (counts and sums); otherwise it means "none", and 0 is wrong
#   seen: a prompt example (Cell 8) shows this pattern
#   amb: above 0 when an image makes the answer ambiguous (a tie, or a building exactly at a threshold or on a
#        midline); such images are not used for the question
W = "FROM buildings WHERE image = :image"
JI = "FROM buildings b JOIN images i ON i.image = b.image WHERE b.image = :image"
D2 = ("(a.centroid_x - b.centroid_x) * (a.centroid_x - b.centroid_x) + "
      "(a.centroid_y - b.centroid_y) * (a.centroid_y - b.centroid_y)")
EPS = 1e-6

def tie_at_max(where, col="area_px"):
    return f"SELECT COUNT(*) > 1 {W} AND {where} AND {col} = (SELECT MAX({col}) {W} AND {where})"

def on_threshold(where, col, value):
    return f"SELECT COUNT(*) {W} AND {where} AND ABS({col} - {value}) < {EPS}"

def on_midline(where, axis):
    size = "width" if axis == "x" else "height"
    return f"SELECT COUNT(*) {JI} AND {where} AND ABS(b.centroid_{axis} - i.{size} / 2.0) < {EPS}"

def pair_on(where, d2):
    return (f"SELECT COUNT(*) FROM buildings a JOIN buildings b ON b.image = a.image "
            f"WHERE a.image = :image AND {where} AND ABS({D2} - {d2}) < {EPS}")

def Q(id, group, gold, phrasings, kind="number", tol=0.0, zero_if_none=False, seen=False, amb=None):
    return dict(id=id, group=group, gold=gold, phrasings=phrasings, kind=kind, tol=tol,
                zero_if_none=zero_if_none, seen=seen, amb=amb, answerable=True)

def U(id, phrasings):
    return dict(id=id, group="unanswerable", gold=None, phrasings=phrasings, kind=None, tol=0.0,
                zero_if_none=False, seen=False, amb=None, answerable=False)

ANSWERABLE = [
    Q("A1", "count", f"SELECT COUNT(*) {W} AND class_name = 'destroyed'",
      ["How many destroyed buildings are there?",
       "How many structures were completely flattened?",
       "I need the number of buildings in this scene that were totally destroyed."],
      zero_if_none=True, seen=True),
    Q("A2", "count", f"SELECT COUNT(*) {W} AND class_name = 'intact'",
      ["How many intact buildings are there?",
       "How many structures are still standing undamaged?",
       "Tell me how many buildings came through the disaster without any damage."],
      zero_if_none=True, seen=True),
    Q("A3", "count", f"SELECT COUNT(*) {W} AND class_name = 'damaged'",
      ["How many damaged buildings are there?",
       "How many structures were partly damaged but are still standing?",
       "What is the number of buildings that have some damage but were not destroyed?"],
      zero_if_none=True, seen=True),
    Q("B1", "total", f"SELECT COUNT(*) {W}",
      ["How many buildings are in the image in total?",
       "What is the total number of structures, whatever their condition?",
       "Count every building in this scene, damaged or not."], zero_if_none=True),
    Q("B2", "total", f"SELECT COUNT(*) {W} AND class_name IN ('damaged', 'destroyed')",
      ["How many buildings are damaged or destroyed?",
       "How many structures were hit by the disaster at all?",
       "Give me the combined number of damaged and destroyed buildings."], zero_if_none=True),
    Q("C1", "percentage", f"SELECT 100.0 * SUM(class_name = 'destroyed') / COUNT(*) {W}",
      ["What percentage of the buildings are destroyed?",
       "What share of the structures were flattened, as a percent?",
       "Out of all the buildings in this scene, what percent were totally destroyed?"], tol=0.5),
    Q("C2", "percentage", f"SELECT 100.0 * SUM(class_name IN ('damaged', 'destroyed')) / COUNT(*) {W}",
      ["What percentage of the buildings are damaged or destroyed?",
       "What percent of the structures show any damage at all?",
       "As a percentage of all buildings here, how many were damaged or destroyed?"], tol=0.5),
    Q("D1", "area", f"SELECT COALESCE(SUM(area_px), 0) {W} AND class_name = 'destroyed'",
      ["What is the total area of the destroyed buildings, in pixels?",
       "How many pixels do the flattened structures cover altogether?",
       "Add up the areas of all destroyed buildings: what is the total in pixels?"],
      zero_if_none=True, seen=True),
    Q("D2", "area", f"SELECT AVG(area_px) {W} AND class_name = 'intact'",
      ["What is the average area of the intact buildings, in pixels?",
       "On average, how big is an undamaged structure, in pixels?",
       "What is the mean area, in pixels, of the buildings that are still intact?"], tol=0.5),
    Q("D3", "area", f"SELECT MAX(area_px) {W} AND class_name = 'damaged'",
      ["What is the area of the largest damaged building, in pixels?",
       "How big is the biggest partly damaged structure, in pixels?",
       "Among the damaged buildings, what is the largest area in pixels?"]),
    Q("D4", "area", f"SELECT MIN(area_px) {W} AND class_name = 'intact'",
      ["What is the area of the smallest intact building, in pixels?",
       "How many pixels does the tiniest undamaged structure cover?",
       "Among the intact buildings, what is the smallest area in pixels?"]),
    Q("E1", "size threshold", f"SELECT COUNT(*) {W} AND area_px > 500",
      ["How many buildings have an area of more than 500 pixels?",
       "How many structures are bigger than 500 pixels?",
       "Count the buildings whose area exceeds 500 pixels."], zero_if_none=True),
    Q("E2", "size threshold", f"SELECT COUNT(*) {W} AND class_name = 'destroyed' AND area_px >= 200",
      ["How many destroyed buildings have an area of at least 200 pixels?",
       "How many flattened structures cover 200 pixels or more?",
       "Count the destroyed buildings that are no smaller than 200 pixels."], zero_if_none=True),
    Q("F1", "distance", f"SELECT COUNT(*) {W} AND class_name = 'intact' AND dist_to_nearest_destroyed_px <= 100",
      ["How many intact buildings are within 100 pixels of a destroyed building?",
       "How many undamaged structures stand 100 pixels or less from a flattened one?",
       "Count the intact buildings that have a destroyed building no more than 100 pixels away."],
      zero_if_none=True, seen=True,
      amb=on_threshold("class_name = 'intact'", "dist_to_nearest_destroyed_px", 100)),
    Q("F2", "no destroyed nearby",
      f"SELECT COUNT(*) {W} AND class_name = 'damaged' "
      f"AND (dist_to_nearest_destroyed_px > 200 OR dist_to_nearest_destroyed_px IS NULL)",
      ["How many damaged buildings have no destroyed building within 200 pixels?",
       "How many partly damaged structures are more than 200 pixels away from every flattened one?",
       "Count the damaged buildings for which no destroyed building is 200 pixels away or closer."],
      zero_if_none=True, amb=on_threshold("class_name = 'damaged'", "dist_to_nearest_destroyed_px", 200)),
    Q("F3", "distance", f"SELECT COUNT(*) {W} AND class_name = 'destroyed' AND dist_to_nearest_destroyed_px <= 50",
      ["How many destroyed buildings have another destroyed building within 50 pixels?",
       "How many flattened structures sit 50 pixels or less from another flattened structure?",
       "Count the destroyed buildings that are no more than 50 pixels from a different destroyed building."],
      zero_if_none=True, seen=True,
      amb=on_threshold("class_name = 'destroyed'", "dist_to_nearest_destroyed_px", 50)),
    Q("F4", "distance",
      f"SELECT dist_to_nearest_destroyed_px {W} AND class_name = 'intact' ORDER BY area_px DESC LIMIT 1",
      ["How far is the largest intact building from the nearest destroyed building, in pixels?",
       "What is the distance in pixels from the biggest undamaged structure to the closest flattened one?",
       "Take the intact building with the largest area: how many pixels away is the nearest destroyed building?"],
      tol=0.051, amb=tie_at_max("class_name = 'intact'")),
    Q("G1", "position", f"SELECT COUNT(*) {JI} AND b.class_name = 'destroyed' AND b.centroid_x < i.width / 2.0",
      ["How many destroyed buildings are in the left half of the image?",
       "How many flattened structures are on the left half of the picture?",
       "Count the destroyed buildings whose centre lies left of the image's vertical midline."],
      zero_if_none=True, amb=on_midline("b.class_name = 'destroyed'", "x")),
    Q("G2", "position",
      f"SELECT COUNT(*) {JI} AND b.centroid_x < i.width / 2.0 AND b.centroid_y < i.height / 2.0",
      ["How many buildings are in the top-left quarter of the image?",
       "How many structures sit in the upper-left quadrant of the picture?",
       "Count the buildings whose centre falls in the top-left quarter of the image."],
      zero_if_none=True, amb=f"SELECT ({on_midline('1', 'x')}) + ({on_midline('1', 'y')})"),
    Q("G3", "position", f"SELECT COUNT(*) {JI} AND b.class_name = 'intact' AND b.centroid_y >= i.height / 2.0",
      ["How many intact buildings are in the bottom half of the image?",
       "How many undamaged structures are in the lower half of the picture?",
       "Count the intact buildings whose centre lies below the image's horizontal midline."],
      zero_if_none=True, amb=on_midline("b.class_name = 'intact'", "y")),
    Q("H1", "ranking", f"SELECT class_name {W} GROUP BY class_name ORDER BY COUNT(*) DESC LIMIT 1",
      ["Which damage class has the most buildings in this image?",
       "What condition are most structures in here: intact, damaged or destroyed?",
       "Is the most common building state in this scene intact, damaged or destroyed?"],
      kind="class",
      amb=f"SELECT COUNT(*) > 1 FROM (SELECT COUNT(*) AS n {W} GROUP BY class_name) "
          f"WHERE n = (SELECT MAX(n) FROM (SELECT COUNT(*) AS n {W} GROUP BY class_name))"),
    Q("H2", "ranking", f"SELECT building_id {W} AND class_name = 'destroyed' ORDER BY area_px DESC LIMIT 1",
      ["What is the ID of the largest destroyed building?",
       "Which flattened structure is the biggest? Give its building ID.",
       "Give the building_id of the destroyed building with the largest area."],
      amb=tie_at_max("class_name = 'destroyed'")),
    Q("H3", "ranking", f"SELECT building_id {W} ORDER BY area_px DESC LIMIT 3",
      ["List the IDs of the three largest buildings, largest first.",
       "Which three structures are the biggest? Give their building IDs from largest to smallest.",
       "Give the building IDs of the 3 buildings with the largest area, in descending order of area."],
      kind="list",
      amb=f"SELECT COUNT(*) - COUNT(DISTINCT area_px) FROM (SELECT area_px {W} ORDER BY area_px DESC LIMIT 4)"),
    Q("I1", "yes/no", f"SELECT EXISTS (SELECT 1 {W} AND class_name = 'destroyed')",
      ["Is there any destroyed building in this image?",
       "Was any structure here completely flattened?",
       "Does this scene contain at least one destroyed building?"], kind="bool"),
    Q("I2", "yes/no",
      f"SELECT (SELECT COUNT(*) {W} AND class_name = 'damaged') > (SELECT COUNT(*) {W} AND class_name = 'destroyed')",
      ["Are there more damaged buildings than destroyed buildings?",
       "Do partly damaged structures outnumber flattened ones?",
       "Is the number of damaged buildings greater than the number of destroyed buildings?"], kind="bool"),
    Q("J1", "pairwise",
      f"SELECT COUNT(*) FROM buildings a WHERE a.image = :image AND a.class_name = 'intact' AND EXISTS "
      f"(SELECT 1 FROM buildings b WHERE b.image = a.image AND b.class_name = 'damaged' AND {D2} <= 2500)",
      ["How many intact buildings have a damaged building within 50 pixels, measured centre to centre?",
       "How many undamaged structures have a partly damaged structure 50 pixels or less away, centre to centre?",
       "Count the intact buildings whose centre is no more than 50 pixels from the centre of some damaged building."],
      zero_if_none=True, amb=pair_on("a.class_name = 'intact' AND b.class_name = 'damaged'", 2500)),
    Q("J2", "pairwise",
      f"SELECT COUNT(*) FROM buildings a JOIN buildings b ON b.image = a.image AND a.building_id < b.building_id "
      f"WHERE a.image = :image AND a.class_name = 'destroyed' AND b.class_name = 'destroyed' AND {D2} < 900",
      ["How many pairs of destroyed buildings are less than 30 pixels apart, centre to centre?",
       "How many pairs of flattened structures have centres closer than 30 pixels to each other?",
       "Count the distinct pairs of destroyed buildings whose centres are under 30 pixels apart."],
      zero_if_none=True,
      amb=pair_on("a.class_name = 'destroyed' AND b.class_name = 'destroyed' AND a.building_id < b.building_id", 900)),
    Q("K1", "combined",
      f"SELECT COUNT(*) {W} AND class_name = 'damaged' AND area_px > 300 AND dist_to_nearest_destroyed_px <= 150",
      ["How many damaged buildings larger than 300 pixels are within 150 pixels of a destroyed building?",
       "How many partly damaged structures bigger than 300 pixels stand 150 pixels or less from a flattened one?",
       "Count the damaged buildings that have an area over 300 pixels and a destroyed building no more than 150 pixels away."],
      zero_if_none=True,
      amb=on_threshold("class_name = 'damaged' AND area_px > 300", "dist_to_nearest_destroyed_px", 150)),
    Q("K2", "combined",
      f"SELECT AVG(b.area_px) {JI} AND b.class_name = 'destroyed' AND b.centroid_x >= i.width / 2.0",
      ["What is the average area of the destroyed buildings in the right half of the image?",
       "On the right half of the picture, how big is a flattened structure on average, in pixels?",
       "Among the destroyed buildings whose centre is in the right half of the image, what is the mean area in pixels?"],
      tol=0.5, amb=on_midline("b.class_name = 'destroyed'", "x")),
]
UNANSWERABLE = [
    U("U1", ["How many people were injured in this area?",
             "What is the number of casualties in this scene?",
             "How many residents were hurt?"]),
    U("U2", ["What is the area of the largest destroyed building in square metres?",
             "How many square metres do the damaged buildings cover in total?",
             "What is the footprint of the biggest intact building in m²?"]),
    U("U3", ["How many roads are blocked by debris?",
             "Which streets are impassable?",
             "How many kilometres of road were damaged?"]),
    U("U4", ["How many trees were knocked down?",
             "How much vegetation was destroyed?",
             "How many fallen trees are there near the buildings?"]),
    U("U5", ["How many of the destroyed buildings are hospitals or schools?",
             "Which damaged buildings are homes and which are shops?",
             "How many destroyed structures were residential?"]),
    U("U6", ["How many floors did the tallest building have?",
             "How tall is the largest intact building?",
             "What is the height of the destroyed buildings?"]),
    U("U7", ["When did the disaster happen?",
             "On what date was this image taken?",
             "How many days after the event was this picture captured?"]),
    U("U8", ["Which destroyed buildings collapsed and which burned down?",
             "For each damaged building, was the damage caused by wind or by water?",
             "How many buildings were damaged by the aftershock rather than the main event?"]),
    U("U9", ["How deep is the flood water around the damaged buildings?",
             "What is the water level at the destroyed buildings?",
             "How high did the flood rise in this scene?"]),
    U("U10", ["How many vehicles are in the image?",
              "How many cars were damaged?",
              "Count the boats in this scene."]),
    U("U11", ["How far is the nearest hospital from the destroyed buildings?",
              "Where is the closest shelter to the damaged buildings?",
              "What is the distance from the destroyed buildings to the nearest fire station?"]),
    U("U12", ["What will it cost to repair the damaged buildings?",
              "What is the estimated economic loss in this scene?",
              "How much money is needed to rebuild the destroyed buildings?"]),
]
assert len({q["id"] for q in ANSWERABLE + UNANSWERABLE}) == len(ANSWERABLE + UNANSWERABLE)
assert all(len(q["phrasings"]) == 3 for q in ANSWERABLE + UNANSWERABLE)
PHRASES = [0, 1, 2]
PHRASING_NAMES = {0: "plain", 1: "responder words", 2: "indirect"}
if SMOKE_TEST:
    ANSWERABLE = [q for q in ANSWERABLE if q["id"] in ("A1", "C1", "F2", "H3")]
    UNANSWERABLE = [q for q in UNANSWERABLE if q["id"] in ("U2", "U8")]
    PHRASES = [0]
QBANK = {q["id"]: q for q in ANSWERABLE + UNANSWERABLE}
print(f"✓ {len(ANSWERABLE)} answerable questions in {len({q['group'] for q in ANSWERABLE})} groups "
      f"({sum(q['seen'] for q in ANSWERABLE)} shown by a prompt example), {len(UNANSWERABLE)} unanswerable, "
      f"{len(PHRASES)} phrasing(s) each")

In [ ]:
# ── Cell 5: Running a query, and reading its result as an answer (pure functions) ──
NONE, AMBIGUOUS = "NONE", "AMBIGUOUS"     # no answer (no rows, or NULL); a version of the table not used

def prepare(conn):
    # sqrt and power where this SQLite build lacks them (models use them for distances), then read-only
    try:
        conn.execute("SELECT sqrt(4), power(2, 3), pow(2, 3)")
    except sqlite3.OperationalError:
        conn.create_function("sqrt", 1, lambda x: None if x is None or x < 0 else math.sqrt(x), deterministic=True)
        for fn in ("power", "pow"):
            conn.create_function(fn, 2, lambda x, y: None if x is None or y is None else math.pow(x, y),
                                 deterministic=True)
    conn.execute("PRAGMA query_only = ON")
    return conn

def run_sql(conn, sql, params=None, timeout_s=30, max_rows=1000):
    """(rows, None), or (None, the error message). A model's own queries get SQL_TIMEOUT_S (Cell 10)."""
    t0 = time.time()
    conn.set_progress_handler(lambda: 1 if time.time() - t0 > timeout_s else 0, 10000)
    try:
        cur = conn.execute(sql) if params is None else conn.execute(sql, params)
        rows = cur.fetchmany(max_rows + 1)
        if len(rows) > max_rows:
            return None, f"the query returned more than {max_rows} rows"
        return [tuple(r) for r in rows], None
    except Exception as e:
        if "interrupted" in str(e):
            return None, f"the query took longer than {timeout_s} s"
        return None, f"{type(e).__name__}: {e}"
    finally:
        conn.set_progress_handler(None, 0)

def _num(v):
    if isinstance(v, (bool, int, float)):
        return float(v)
    if isinstance(v, str):
        try:
            return float(v.strip().rstrip("%").strip())
        except ValueError:
            return None
    return None

def _truth(v):
    n = _num(v)
    if n is not None:
        return n != 0
    s = str(v).strip().lower()
    return True if s in ("yes", "true") else False if s in ("no", "false") else None

def _value(v, q):
    """One cell read as an answer to q; None when it can't be one."""
    if v is None:
        if q["kind"] == "bool":
            return False
        return 0.0 if q["zero_if_none"] else NONE
    if q["kind"] == "number":
        return _num(v)
    if q["kind"] == "bool":
        return _truth(v)
    n = _num(v)                                    # a class: its name, or its class_id
    return CLASS_BY_ID[int(n)] if n is not None and n in (1, 2, 3) else str(v).strip().lower()

def answer_forms(rows, q, lenient=False):
    """The answers a result can be read as. Strict: one row of one column (one column for a list).
    Lenient: any one column of the result."""
    if rows is None:
        return []
    if q["kind"] == "list":
        if not rows:
            return [()]
        cols = range(len(rows[0])) if lenient else ([0] if len(rows[0]) == 1 else [])
        return [tuple(_num(r[c]) for r in rows) for c in cols]
    if not rows:
        return [_value(None, q)]
    if len(rows) != 1:
        return []
    cells = rows[0] if (lenient or len(rows[0]) == 1) else ()
    return [_value(v, q) for v in cells]

def matches(rows, gold, q, lenient=False):
    for a in answer_forms(rows, q, lenient):
        if a is None:
            continue
        if q["kind"] == "number" and isinstance(a, float) and isinstance(gold, float):
            if abs(a - gold) <= q["tol"] + 1e-9:
                return True
        elif a == gold:
            return True
    return False

def trivial(a):
    # a zero, none, no or empty answer: what the "edge" instances have
    return a == NONE or a == () or a is False or (isinstance(a, float) and a == 0.0)

print("✓ Query and answer helpers ready")

In [ ]:
# ── Cell 6: The test suite (five versions of the table), the instances and their reference answers ──
# A query can match the reference on one table by chance (Zhong, Yu & Klein 2020), so every answer also runs on
# four redrawn versions. Only the instances' images change in them; every other image keeps its real rows.
from scipy.spatial import cKDTree

VARIANTS = ["real", "classes redrawn", "geometry redrawn", "no destroyed", "no buildings"]
COLNAMES = list(COLUMN_NOTES)
INSERT = f"INSERT INTO buildings ({', '.join(COLNAMES)}) VALUES ({', '.join('?' * len(COLNAMES))})"

def nearest_destroyed(rows):
    # centre-to-centre distance to the nearest OTHER destroyed building (H2's definition), rounded to 0.1 px
    out = [None] * len(rows)
    xy = rows[["centroid_x", "centroid_y"]].to_numpy(float)
    dest = rows["class_id"].to_numpy() == 3
    if not dest.any():
        return out
    tree = cKDTree(xy[dest])
    for r in range(len(rows)):
        if not dest[r]:
            out[r] = round(float(tree.query(xy[r], k=1)[0]), 1)
        elif dest.sum() > 1:
            out[r] = round(float(tree.query(xy[r], k=2)[0][1]), 1)    # the nearest hit is the building itself
    return out

def sql_value(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    return v.item() if isinstance(v, np.generic) else v

def make_variant(name, images, rng):
    conn = sqlite3.connect(":memory:")
    src = sqlite3.connect(f"file:{H2_DB}?mode=ro", uri=True)
    src.backup(conn)
    src.close()
    for img in (images if name != "real" else []):
        rows = BUILDINGS[BUILDINGS["image"] == img].copy()
        w, h = SIZE[img]
        n = len(rows)
        if name == "classes redrawn":
            rows["class_id"] = rng.choice([1, 2, 3], size=n, p=[0.45, 0.30, 0.25])
        elif name == "geometry redrawn":
            area = rng.permutation(rows["area_px"].to_numpy()) * rng.uniform(0.5, 2.0, n)
            rows["area_px"] = np.maximum(1, np.round(area)).astype(int)
            rows["centroid_x"] = np.round(rng.uniform(0, w, n), 1)
            rows["centroid_y"] = np.round(rng.uniform(0, h, n), 1)
            side = np.maximum(1, np.round(np.sqrt(rows["area_px"].to_numpy()))).astype(int)
            rows["bbox_w"], rows["bbox_h"] = side, side
            rows["bbox_x"] = np.clip(np.round(rows["centroid_x"].to_numpy() - side / 2), 0, w - 1).astype(int)
            rows["bbox_y"] = np.clip(np.round(rows["centroid_y"].to_numpy() - side / 2), 0, h - 1).astype(int)
        elif name == "no destroyed":
            rows = rows[rows["class_id"] != 3].copy()
        else:                                                         # no buildings
            rows = rows.iloc[0:0].copy()
        rows["class_name"] = rows["class_id"].map(CLASS_BY_ID)
        rows["dist_to_nearest_destroyed_px"] = nearest_destroyed(rows)
        conn.execute("DELETE FROM buildings WHERE image = ?", (img,))
        conn.executemany(INSERT, [tuple(sql_value(v) for v in r) for r in rows[COLNAMES].itertuples(index=False)])
    conn.commit()
    return prepare(conn)

def gold_answer(conn, q, image):
    """The reference answer in canonical form, or AMBIGUOUS."""
    if q["amb"]:
        rows, err = run_sql(conn, q["amb"], {"image": image})
        assert err is None, f"{q['id']} ambiguity query: {err}"
        if rows and rows[0][0]:
            return AMBIGUOUS
    rows, err = run_sql(conn, q["gold"], {"image": image})
    assert err is None, f"{q['id']} reference query: {err}"
    forms = answer_forms(rows, q)
    assert len(forms) == 1 and forms[0] is not None, f"{q['id']}: unexpected reference result {rows}"
    return forms[0]

t0 = time.time()
DBS = {"real": make_variant("real", [], None)}
with_buildings = sorted(BUILDINGS["image"].unique())
POOL = sorted(np.random.default_rng(SEED).choice(with_buildings, size=min(POOL_IMAGES, len(with_buildings)),
                                                 replace=False).tolist())
INSTANCES = []
for k, q in enumerate(ANSWERABLE):
    typical, edge = [], []
    for img in POOL:
        a = gold_answer(DBS["real"], q, img)
        if a != AMBIGUOUS:
            (edge if trivial(a) else typical).append(img)
    r = np.random.default_rng(SEED + 1 + k)
    n_edge = 1 if (edge and IMAGES_PER_QUESTION > 1) else 0
    pick = list(r.choice(typical, size=min(IMAGES_PER_QUESTION - n_edge, len(typical)), replace=False)) if typical else []
    pick += list(r.choice(edge, size=n_edge, replace=False)) if n_edge else []
    if not pick:
        print(f"⚠ {q['id']}: no usable image in the pool, left out")
    INSTANCES += [{"inst": f"{q['id']}@{img}", "qid": q["id"], "image": str(img)} for img in pick]
for k, q in enumerate(UNANSWERABLE):
    img = str(np.random.default_rng(SEED + 101 + k).choice(POOL))
    INSTANCES.append({"inst": f"{q['id']}@{img}", "qid": q["id"], "image": img})

SEL = sorted({i["image"] for i in INSTANCES})
variant_rng = np.random.default_rng(SEED + 7)
for v in VARIANTS[1:]:
    DBS[v] = make_variant(v, SEL, variant_rng)
GOLD = {i["inst"]: {v: gold_answer(DBS[v], QBANK[i["qid"]], i["image"]) for v in VARIANTS}
        for i in INSTANCES if QBANK[i["qid"]]["answerable"]}

ITEMS = pd.DataFrame([{"item": f"{i['qid']}.p{p}@{i['image']}", "inst": i["inst"], "qid": i["qid"],
                       "group": QBANK[i["qid"]]["group"], "seen": QBANK[i["qid"]]["seen"],
                       "answerable": QBANK[i["qid"]]["answerable"], "phrasing": p, "image": i["image"],
                       "question": QBANK[i["qid"]]["phrasings"][p]} for i in INSTANCES for p in PHRASES])
assert ITEMS["item"].is_unique
ITEMS.to_csv(OUT_DIR / "items.csv", index=False)
(OUT_DIR / "gold.json").write_text(json.dumps(GOLD, indent=1), encoding="utf-8")
left_out = Counter(k.split("@")[0] for k, g in GOLD.items() for v in VARIANTS if g[v] == AMBIGUOUS)
print(f"✓ {len(INSTANCES)} instances on {len(SEL)} images, "
      f"{sum(trivial(g['real']) for g in GOLD.values())} of them with a zero or none answer; "
      f"{len(ITEMS)} items ({int(ITEMS['answerable'].sum())} answerable), {(time.time() - t0) / 60:.1f} min")
print(f"  Versions left out as ambiguous: {sum(left_out.values())} {dict(left_out)}")

In [ ]:
# ── Cell 7: Self-checks: the grader, and whether the test suite catches near-miss queries ──
_n = {"kind": "number", "tol": 0.0, "zero_if_none": True}
_none = {"kind": "number", "tol": 0.0, "zero_if_none": False}
_pct = {"kind": "number", "tol": 0.5, "zero_if_none": False}
_b = {"kind": "bool", "tol": 0.0, "zero_if_none": False}
_c = {"kind": "class", "tol": 0.0, "zero_if_none": False}
_l = {"kind": "list", "tol": 0.0, "zero_if_none": False}
assert matches([(3,)], 3.0, _n) and not matches([(3, 4)], 3.0, _n) and matches([(3, 4)], 3.0, _n, lenient=True)
assert matches([], 0.0, _n) and matches([(None,)], 0.0, _n) and not matches([(1,), (2,)], 1.0, _n)
assert matches([], NONE, _none) and matches([(None,)], NONE, _none) and not matches([(0,)], NONE, _none)
assert matches([(66.7,)], 200 / 3, _pct) and matches([("66.67%",)], 200 / 3, _pct) and not matches([(66,)], 200 / 3, _pct)
assert matches([("yes",)], True, _b) and matches([(0,)], False, _b) and matches([], False, _b)
assert not matches([("maybe",)], True, _b)
assert matches([(3,)], "destroyed", _c) and matches([("Destroyed",)], "destroyed", _c) and not matches([(2,)], "destroyed", _c)
assert matches([(5,), (2,), (9,)], (5.0, 2.0, 9.0), _l) and not matches([(2,), (5,), (9,)], (5.0, 2.0, 9.0), _l)
assert not matches([(5, 90), (2, 80), (9, 70)], (5.0, 2.0, 9.0), _l)
assert matches([(5, 90), (2, 80), (9, 70)], (5.0, 2.0, 9.0), _l, lenient=True)
assert not matches(None, 3.0, _n)

def on_suite(sql, qid):
    """Per instance of qid: does sql (written with {img}) match the reference on every usable version?"""
    out = []
    for i in INSTANCES:
        if i["qid"] == qid:
            s = sql.format(img=i["image"])
            out.append(all(g == AMBIGUOUS or matches(run_sql(DBS[v], s)[0], g, QBANK[qid])
                           for v, g in GOLD[i["inst"]].items()))
    return out

IMG = "'{img}'"
EQUIVALENT = {   # correct rewrites, written differently from the reference: each must pass everywhere
    "C1": f"SELECT ROUND(CAST(COUNT(CASE WHEN class_name = 'destroyed' THEN 1 END) AS REAL) * 100 / COUNT(*), 1) "
          f"FROM buildings WHERE image = {IMG}",
    "F2": f"SELECT COUNT(*) FROM buildings WHERE image = {IMG} AND class_name = 'damaged' "
          f"AND COALESCE(dist_to_nearest_destroyed_px, 1e9) > 200",
    "J1": f"SELECT COUNT(DISTINCT a.building_id) FROM buildings a JOIN buildings b ON a.image = b.image "
          f"WHERE a.image = {IMG} AND a.class_name = 'intact' AND b.class_name = 'damaged' "
          f"AND SQRT(POWER(a.centroid_x - b.centroid_x, 2) + POWER(a.centroid_y - b.centroid_y, 2)) <= 50",
    "I1": f"SELECT CASE WHEN COUNT(*) > 0 THEN 'yes' ELSE 'no' END FROM buildings WHERE image = {IMG} "
          f"AND class_name = 'destroyed'",
    "H1": f"SELECT class_id FROM buildings WHERE image = {IMG} GROUP BY class_id ORDER BY COUNT(*) DESC LIMIT 1",
    "D3": f"SELECT area_px FROM buildings WHERE image = {IMG} AND class_name = 'damaged' ORDER BY area_px DESC LIMIT 1",
    "G3": f"SELECT COUNT(*) FROM buildings WHERE image = {IMG} AND class_name = 'intact' "
          f"AND centroid_y > (SELECT height FROM images WHERE image = {IMG}) / 2.0",
}
MUTANT = {       # near misses a model could write: the suite should catch each one somewhere
    "C1": f"SELECT 100 * SUM(class_name = 'destroyed') / COUNT(*) FROM buildings WHERE image = {IMG}",   # integer division
    "F2": f"SELECT COUNT(*) FROM buildings WHERE image = {IMG} AND class_name = 'damaged' "
          f"AND dist_to_nearest_destroyed_px > 200",                                                     # drops "none at all"
    "G3": f"SELECT COUNT(*) FROM buildings WHERE image = {IMG} AND class_name = 'intact' "
          f"AND centroid_y < (SELECT height FROM images WHERE image = {IMG}) / 2.0",                     # y axis flipped
    "J2": f"SELECT COUNT(*) FROM buildings a JOIN buildings b ON a.image = b.image AND a.building_id <> b.building_id "
          f"WHERE a.image = {IMG} AND a.class_name = 'destroyed' AND b.class_name = 'destroyed' AND {D2} < 900",  # each pair twice
    "H3": f"SELECT building_id FROM buildings WHERE image = {IMG} ORDER BY area_px ASC LIMIT 3",          # smallest first
    "D2": f"SELECT AVG(area_px) FROM buildings WHERE image = {IMG}",                                    # every class
}
bad = [f"{qid} reference" for qid, q in QBANK.items() if q["answerable"]
       and not all(on_suite(q["gold"].replace(":image", IMG), qid))]
bad += [f"{qid} rewrite" for qid, sql in EQUIVALENT.items() if qid in QBANK and not all(on_suite(sql, qid))]
assert not bad, f"Correct queries judged wrong: {bad}"
caught = {qid: not all(on_suite(sql, qid)) for qid, sql in MUTANT.items() if qid in QBANK}
real_only = [qid for qid, sql in MUTANT.items() if qid in QBANK and
             all(matches(run_sql(DBS["real"], sql.format(img=i["image"]))[0], GOLD[i["inst"]]["real"], QBANK[qid])
                 for i in INSTANCES if i["qid"] == qid)]
print("✓ Grader self-checks passed; every reference query and correct rewrite passes on all versions of the table")
print(f"  Near-miss queries caught by the suite: {caught}")
print(f"  ... that the real table alone would have passed: {real_only}")
if not SMOKE_TEST:
    assert all(caught.values()), f"The suite misses near-miss queries: {[k for k, v in caught.items() if not v]}"

In [ ]:
# ── Cell 8: The prompt, fixed 22 Oct before any run and the same for every model; reading a reply ──
RULES = [
    "Reply with one SQLite SELECT statement only: no explanation, no markdown.",
    "Always filter on the image named in the message: WHERE image = '<that file name>'.",
    "The image name is only an identifier: do not read facts from it.",
    "Positions, distances and areas are in pixels; the ground size of a pixel is unknown.",
    "A building's position is its centre (centroid_x, centroid_y); distances between buildings are centre to centre.",
    "If the tables cannot answer the question, reply exactly: UNANSWERABLE",
]
EXAMPLES = [   # placeholder images; the patterns of the first three are the ones marked "seen" in the question bank
    ("example_a.png", "How many destroyed buildings are there?",
     "SELECT COUNT(*) FROM buildings WHERE image = 'example_a.png' AND class_name = 'destroyed'"),
    ("example_b.png", "How many intact buildings are within 150 pixels of a destroyed building?",
     "SELECT COUNT(*) FROM buildings WHERE image = 'example_b.png' AND class_name = 'intact' "
     "AND dist_to_nearest_destroyed_px <= 150"),
    ("example_c.png", "What is the total area of the damaged buildings?",
     "SELECT COALESCE(SUM(area_px), 0) FROM buildings WHERE image = 'example_c.png' AND class_name = 'damaged'"),
    ("example_d.png", "What is the name of the town in this image?", "UNANSWERABLE"),
]
SYSTEM_PROMPT = ("You answer questions about ONE satellite image of a disaster area by writing a single SQLite "
                 "query over these tables.\n\n" + SCHEMA_SQL + "\n\nRules:\n" + "\n".join(f"- {r}" for r in RULES)
                 + "\n\nExamples (the file names are placeholders):\n\n"
                 + "\n\n".join(f"Image: {im}\nQuestion: {q}\nSQL: {s}" for im, q, s in EXAMPLES))
(OUT_DIR / "prompt.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")

def user_prompt(question, image):
    return f"Image: {image}\nQuestion: {question}\nSQL:"

FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|vacuum|reindex)\b", re.I)
START = re.compile(r"\bwith\s+\w+(\s*\([^)]*\))?\s+as\s*\(|\bselect\b", re.I)

def parse_reply(text):
    """('sql', query) | ('abstain', '') | ('invalid', reason)"""
    t = text.strip()
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", t, re.S | re.I)
    if m:
        t = m.group(1).strip()
    t = re.sub(r"^\s*SQL:\s*", "", t, flags=re.I)
    if re.match(r"^\W*UNANSWERABLE\b", t, re.I):
        return "abstain", ""
    m = START.search(t)
    if not m:
        return ("abstain", "") if "UNANSWERABLE" in t.upper() else ("invalid", "the reply holds no SELECT statement")
    sql = re.split(r"\n\s*\n", t[m.start():])[0]      # stop at a blank line: an explanation may follow
    return "sql", sql.split(";")[0].strip()

def guard(sql):
    if not re.match(r"^(select|with)\b", sql, re.I):
        return "the reply is not a SELECT statement"
    if FORBIDDEN.search(sql):
        return "the query uses a keyword that changes data"
    return None

def feedback(kind, error):
    if kind == "invalid":
        return f"{error}. Reply with one SQLite SELECT statement only, or UNANSWERABLE."
    return f"That query failed: {error}. Reply with one corrected SQLite SELECT statement, or UNANSWERABLE."

assert parse_reply("SELECT COUNT(*) FROM buildings;") == ("sql", "SELECT COUNT(*) FROM buildings")
assert parse_reply("```sql\nSELECT 1\n```") == ("sql", "SELECT 1")
assert parse_reply("UNANSWERABLE") == ("abstain", "") and parse_reply("SQL: UNANSWERABLE.") == ("abstain", "")
assert parse_reply("Here is the query:\nSELECT 2 FROM images\n\nIt counts rows.") == ("sql", "SELECT 2 FROM images")
assert parse_reply("WITH t AS (SELECT 1) SELECT * FROM t")[1].startswith("WITH t AS")
assert parse_reply("These tables cannot say: UNANSWERABLE") == ("abstain", "")
assert parse_reply("I don't know.")[0] == "invalid"
assert guard("DELETE FROM buildings") and guard("SELECT 1; DROP TABLE images") and guard("SELECT 1") is None
print(f"✓ Prompt fixed ({len(SYSTEM_PROMPT):,} characters); reply parsing checked\n")
print(SYSTEM_PROMPT)

## Run the models
Each configuration answers every item: a first try, then up to 2 retries for failed queries, each with the error message. Before that, 24 items are answered one at a time, as the deployed system would answer them, to measure latency and peak memory. The same items are then answered in a batch, to check that batching doesn't change the replies.

In [ ]:
# ── Cell 9: The candidate models: 4-bit or fp16, greedy decoding in batches or one at a time ──
from transformers import (AutoModelForCausalLM, AutoProcessor, AutoTokenizer, BitsAndBytesConfig,
                          GenerationConfig, Qwen2_5_VLForConditionalGeneration)

class LLM:
    def __init__(self, hf_id, kind, precision):
        self.id, self.kind, self.precision = hf_id, kind, precision
        if kind == "vl":    # a Qwen2.5-VL: only its language side is used, so no images are passed
            self.tmpl = AutoProcessor.from_pretrained(hf_id, token=hf_token())
            self.tok, cls = self.tmpl.tokenizer, Qwen2_5_VLForConditionalGeneration
        else:
            self.tmpl = self.tok = AutoTokenizer.from_pretrained(hf_id, token=hf_token())
            cls = AutoModelForCausalLM
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        if precision == "nf4":    # one GPU, as on a low-resource device
            kw = {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                            bnb_4bit_compute_dtype=torch.float16,
                                                            bnb_4bit_use_double_quant=True),
                  "device_map": {"": 0}}
        else:                     # fp16: a 7B (about 16 GB) needs both T4s; the smaller models fit on one
            kw = {"device_map": "auto" if "7b" in hf_id.lower() else {"": 0}}
        self.model = cls.from_pretrained(hf_id, dtype=torch.float16, token=hf_token(), **kw).eval()
        self.footprint_gb = self.model.get_memory_footprint() / 1e9
        eos = {self.tok.convert_tokens_to_ids("<|im_end|>"), self.tok.eos_token_id}
        g = self.model.generation_config.eos_token_id
        eos |= set(g if isinstance(g, (list, tuple)) else [g])
        eos.discard(None)
        # greedy and without a repetition penalty for every model (some defaults penalise repeats, which SQL needs)
        self.gen = GenerationConfig(do_sample=False, max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=1.0,
                                    pad_token_id=self.tok.pad_token_id, eos_token_id=sorted(eos))

    def text(self, messages):
        if self.kind == "vl":
            messages = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]
        return self.tmpl.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def generate(self, conversations, batch_size):
        texts = [self.text(m) for m in conversations]
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   # similar lengths together: less padding
        outs = [None] * len(texts)
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            enc = self.tok([texts[i] for i in idx], return_tensors="pt", padding=True).to(self.model.device)
            gen = self.model.generate(**enc, generation_config=self.gen)
            for i, row in zip(idx, gen[:, enc["input_ids"].shape[1]:]):
                outs[i] = self.tok.decode(row, skip_special_tokens=True)
        return outs

    @torch.no_grad()
    def timed(self, conversation):
        # one question at a time, as the deployed system answers: (reply, seconds, new tokens)
        enc = self.tok(self.text(conversation), return_tensors="pt").to(self.model.device)
        torch.cuda.synchronize()
        t0 = time.time()
        gen = self.model.generate(**enc, generation_config=self.gen)
        torch.cuda.synchronize()
        new = gen[0, enc["input_ids"].shape[1]:]
        return self.tok.decode(new, skip_special_tokens=True), time.time() - t0, int(len(new))

    def free(self):
        del self.model
        gc.collect()
        torch.cuda.empty_cache()

def drop_download(hf_id):
    from huggingface_hub import constants
    shutil.rmtree(Path(constants.HF_HUB_CACHE) / ("models--" + hf_id.replace("/", "--")), ignore_errors=True)

print("✓ Model wrapper ready")

In [ ]:
# ── Cell 10: Every configuration answers every item (a re-run in the same session continues) ──
ANSWERS, RUNS = OUT_DIR / "answers.jsonl", OUT_DIR / "runs.jsonl"
done, runs = {}, {}
if ANSWERS.exists():
    for line in ANSWERS.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        done[(r["config"], r["item"])] = r
if RUNS.exists():
    for line in RUNS.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        runs[r["config"]] = r

def conversation(question, image):
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt(question, image)}]

def answer_all(llm, items, batch):
    # a first try, then up to MAX_SQL_RETRIES rounds for the failed queries, each with its error message
    state = {it["item"]: {"conv": conversation(it["question"], it["image"]), "attempts": []} for it in items}
    pending = list(state)
    for attempt in range(MAX_SQL_RETRIES + 1):
        if not pending:
            break
        replies = llm.generate([state[i]["conv"] for i in pending], batch)
        retry = []
        for i, raw in zip(pending, replies):
            kind, x = parse_reply(raw)
            err = x if kind == "invalid" else ""
            if kind == "sql":
                err = guard(x) or run_sql(DBS["real"], x, timeout_s=SQL_TIMEOUT_S)[1] or ""
            state[i]["attempts"].append({"raw": raw, "kind": kind, "sql": x if kind == "sql" else "", "error": err})
            if err and attempt < MAX_SQL_RETRIES:
                state[i]["conv"] = state[i]["conv"] + [{"role": "assistant", "content": raw},
                                                       {"role": "user", "content": feedback(kind, err)}]
                retry.append(i)
        pending = retry
    return state

ITEM_LIST = ITEMS.to_dict("records")
LAT_IDS = ITEMS.sample(min(LATENCY_ITEMS, len(ITEMS)), random_state=SEED)["item"].tolist()
LAT = {it["item"]: it for it in ITEM_LIST if it["item"] in set(LAT_IDS)}
t_run = time.time()
for n, (name, hf_id, kind, role, prec) in enumerate(CONFIGS):
    cfg = f"{name} {prec}"
    todo = [it for it in ITEM_LIST if (cfg, it["item"]) not in done]
    last_of_model = all(c[0] != name for c in CONFIGS[n + 1:])
    if todo and (time.time() - t_run) / 3600 > TIME_BUDGET_H:
        print(f"⚠ {cfg} skipped: past the {TIME_BUDGET_H} h budget")
        runs.setdefault(cfg, {"config": cfg, "skipped": "time budget"})
        todo = []
    llm = None
    if todo:
        t0 = time.time()
        try:
            llm = LLM(hf_id, kind, prec)
        except Exception as e:
            print(f"⚠ {cfg} could not load: {type(e).__name__}: {e}")
            runs[cfg] = {"config": cfg, "skipped": f"load failed: {type(e).__name__}: {str(e)[:200]}"}
    if llm is not None:
        load_s = time.time() - t0
        # one at a time, as deployed: latency and peak memory; then the same items batched, to check agreement
        for d in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(d)
        one = [llm.timed(conversation(LAT[i]["question"], LAT[i]["image"])) for i in LAT_IDS]
        peak_gb = sum(torch.cuda.max_memory_allocated(d) for d in range(torch.cuda.device_count())) / 1e9
        batched = llm.generate([conversation(LAT[i]["question"], LAT[i]["image"]) for i in LAT_IDS], BATCH_SIZE)
        agree = float(np.mean([a[0].strip() == b.strip() for a, b in zip(one, batched)])) if one else 1.0
        batch = BATCH_SIZE if agree >= AGREE_MIN else 1
        if batch == 1:
            print(f"⚠ {cfg}: batched replies match one-at-a-time replies on only {agree:.0%}, so it decodes one at a time")
        t1 = time.time()
        state = answer_all(llm, todo, batch)
        answer_s = time.time() - t1
        with open(ANSWERS, "a", encoding="utf-8") as f:
            for i, s in state.items():
                row = {"config": cfg, "item": i, "attempts": s["attempts"]}
                done[(cfg, i)] = row
                f.write(json.dumps(row) + "\n")
        secs, toks = [s for _, s, _ in one], [t for _, _, t in one]
        runs[cfg] = {"config": cfg, "model": hf_id, "kind": kind, "role": role, "precision": prec,
                     "footprint_gb": round(llm.footprint_gb, 2), "peak_gb_one_at_a_time": round(peak_gb, 2),
                     "latency_median_s": round(float(np.median(secs)), 2),
                     "latency_p90_s": round(float(np.percentile(secs, 90)), 2),
                     "tokens_per_s": round(sum(toks) / max(sum(secs), 1e-9), 1),
                     "batch_agreement": round(agree, 3), "batch_size_used": batch,
                     "load_min": round(load_s / 60, 1), "answer_min": round(answer_s / 60, 1)}
        with open(RUNS, "a", encoding="utf-8") as f:
            f.write(json.dumps(runs[cfg]) + "\n")
        print(f"✓ {cfg}: {len(state)} items in {answer_s / 60:.1f} min (batch {batch}, agreement {agree:.0%}), "
              f"{llm.footprint_gb:.1f} GB, median {np.median(secs):.1f} s per question one at a time")
        llm.free()
        del llm
    if last_of_model and HF_CACHE_CLEANUP:
        drop_download(hf_id)
print(f"\n✓ {len(done):,} answers ({(time.time() - t_run) / 60:.1f} min)")

## Grade and decide
Every final query runs on the five versions of the table, and an answer is correct only if it matches the reference on all of them (test-suite accuracy). Matching on the real table alone is reported too, to show what the suite catches.

In [ ]:
# ── Cell 11: Grade every answer on all versions of the table; accuracy, refusals, reliability, intervals ──
IT = {it["item"]: it for it in ITEM_LIST}
CFG_NAMES = [f"{c[0]} {c[4]}" for c in CONFIGS]

def grade(att, item):
    """(outcome, matches on the real table, test-suite correct, lenient test-suite correct)"""
    q = QBANK[item["qid"]]
    if att["kind"] == "abstain":
        return "refused", False, False, False
    if att["kind"] != "sql" or att["error"]:
        return "failed", False, False, False       # the system shows an error, not a number
    if not q["answerable"]:
        return "answered", False, False, False
    strict, loose, real_ok = [], [], False
    for v, g in GOLD[item["inst"]].items():
        if g == AMBIGUOUS:
            continue
        rows, _ = run_sql(DBS[v], att["sql"])
        strict.append(matches(rows, g, q))
        loose.append(matches(rows, g, q, lenient=True))
        real_ok = real_ok or (v == "real" and strict[-1])
    return ("correct" if all(strict) else "wrong"), real_ok, all(strict), all(loose)

rows = []
for (cfg, iid), r in done.items():
    if cfg not in CFG_NAMES or iid not in IT:
        continue
    it = IT[iid]
    out, ex, ts, tsl = grade(r["attempts"][-1], it)
    rows.append({"config": cfg, **{k: it[k] for k in ("item", "inst", "qid", "group", "seen", "answerable", "phrasing")},
                 "outcome": out, "ex_real": ex, "ts": ts, "ts_lenient": tsl,
                 "first_ts": grade(r["attempts"][0], it)[2], "attempts": len(r["attempts"]),
                 "final_sql": r["attempts"][-1]["sql"], "final_reply": r["attempts"][-1]["raw"][:300]})
GR = pd.DataFrame(rows)
if GR.empty:
    raise RuntimeError("No answers to grade: see Cell 10's messages.")
for c in sorted({0, RS_PENALTY}):
    GR[f"rs{c}"] = [((1 if o == "correct" else -c if o == "wrong" else 0) if a else (-c if o == "answered" else 1))
                    for o, a in zip(GR["outcome"], GR["answerable"])]
GR.to_csv(OUT_DIR / "graded.csv", index=False)

ANS_Q = sorted(ITEMS.loc[ITEMS["answerable"], "qid"].unique())
ALL_Q = sorted(ITEMS["qid"].unique())
_draws = {}

def boot(g, col, universe):
    # cluster bootstrap: whole questions (all their phrasings and images) resampled with replacement;
    # the same draws for every configuration, so differences between configurations are paired
    key = tuple(universe)
    if key not in _draws:
        _draws[key] = np.random.default_rng(SEED).integers(0, len(universe), (BOOTSTRAP, len(universe)))
    s = g.groupby("qid")[col].sum().reindex(universe, fill_value=0).to_numpy(float)
    n = g.groupby("qid")[col].count().reindex(universe, fill_value=0).to_numpy(float)
    m = _draws[key]
    return s[m].sum(1) / np.maximum(n[m].sum(1), 1)

def ci(stats):
    lo, hi = np.percentile(100 * stats, [2.5, 97.5])
    return f"[{lo:.1f}, {hi:.1f}]"

def pct(s):
    return round(100 * float(np.mean(np.asarray(s, float))), 1) if len(s) else float("nan")

summary = []
for cfg in [c for c in CFG_NAMES if c in set(GR["config"])]:
    g = GR[GR["config"] == cfg]
    a, u = g[g["answerable"]], g[~g["answerable"]]
    rr = runs.get(cfg, {})
    rs = f"rs{RS_PENALTY}"
    row = {"config": cfg, "role": rr.get("role", ""), "answerable_items": len(a), "unanswerable_items": len(u),
           "ts_acc_%": pct(a["ts"]), "ts_acc_ci95": ci(boot(a, "ts", ANS_Q)),
           "ex_real_acc_%": pct(a["ex_real"]), "lenient_ts_acc_%": pct(a["ts_lenient"]),
           "first_try_ts_acc_%": pct(a["first_ts"]),
           "seen_acc_%": pct(a.loc[a["seen"], "ts"]), "unseen_acc_%": pct(a.loc[~a["seen"], "ts"]),
           "all_phrasings_right_%": pct(a.groupby("inst")["ts"].all()),
           "wrong_%": pct(a["outcome"] == "wrong"), "refused_%": pct(a["outcome"] == "refused"),
           "failed_%": pct(a["outcome"] == "failed"),
           "unanswerable_refused_%": pct(u["outcome"] != "answered"),
           "RS0": round(100 * g["rs0"].mean(), 1), f"RS{RS_PENALTY}": round(100 * g[rs].mean(), 1),
           f"RS{RS_PENALTY}_ci95": ci(boot(g, rs, ALL_Q)),
           "footprint_gb": rr.get("footprint_gb"),
           "extra_memory_gb": 0.0 if rr.get("kind") == "vl" else rr.get("footprint_gb"),
           "peak_gb_one_at_a_time": rr.get("peak_gb_one_at_a_time"), "latency_median_s": rr.get("latency_median_s"),
           "latency_p90_s": rr.get("latency_p90_s"), "tokens_per_s": rr.get("tokens_per_s"),
           "batch_agreement": rr.get("batch_agreement"), "batch_size_used": rr.get("batch_size_used")}
    row["passes"] = bool(row["ts_acc_%"] >= PASS_TS_ACC and row[f"RS{RS_PENALTY}"] > 0)
    summary.append(row)
SUMMARY = pd.DataFrame(summary)
SUMMARY.to_csv(OUT_DIR / "summary.csv", index=False)

A = GR[GR["answerable"]].assign(ts=lambda d: d["ts"].astype(float))
order = [c for c in CFG_NAMES if c in set(GR["config"])]
BY_GROUP = A.pivot_table(index="group", columns="config", values="ts", aggfunc="mean").mul(100).round(1)[order]
BY_GROUP.insert(0, "items", ITEMS[ITEMS["answerable"]].groupby("group").size())
BY_PHRASING = A.pivot_table(index="phrasing", columns="config", values="ts", aggfunc="mean").mul(100).round(1)[order]
BY_PHRASING.index = [PHRASING_NAMES[p] for p in BY_PHRASING.index]
UA = GR[~GR["answerable"]].assign(refused=lambda d: (d["outcome"] != "answered").astype(float))
BY_UNANSWERABLE = (UA.pivot_table(index="qid", columns="config", values="refused", aggfunc="mean").mul(100).round(1)[order]
                   if len(UA) else pd.DataFrame())
BY_GROUP.to_csv(OUT_DIR / "by_group.csv")
BY_PHRASING.to_csv(OUT_DIR / "by_phrasing.csv")
BY_UNANSWERABLE.to_csv(OUT_DIR / "by_unanswerable.csv")

PAIRS = ([(f"{m[0]} nf4", f"{m[0]} fp16") for m in MODELS]
         + [("our-7B nf4", "Coder-3B nf4"), ("VL-3B nf4", "Coder-1.5B nf4"), ("Coder-1.5B nf4", "Coder-3B nf4"),
            ("VL-3B nf4", "our-7B nf4")])
pairs = []
for x, y in PAIRS:
    ax, ay = A[A["config"] == x], A[A["config"] == y]
    if ax.empty or ay.empty or set(ax["item"]) != set(ay["item"]):
        continue
    lo, hi = np.percentile(100 * (boot(ax, "ts", ANS_Q) - boot(ay, "ts", ANS_Q)), [2.5, 97.5])
    pairs.append({"a": x, "b": y, "ts_acc_a_%": pct(ax["ts"]), "ts_acc_b_%": pct(ay["ts"]),
                  "difference_pts": round(pct(ax["ts"]) - pct(ay["ts"]), 1), "ci95": f"[{lo:.1f}, {hi:.1f}]",
                  "clear_difference": bool(lo > 0 or hi < 0)})
PAIRS_DF = pd.DataFrame(pairs)
PAIRS_DF.to_csv(OUT_DIR / "pairwise.csv", index=False)

pd.set_option("display.width", 250)
SHOW = ["config", "ts_acc_%", "ts_acc_ci95", "ex_real_acc_%", "first_try_ts_acc_%", "unseen_acc_%",
        "all_phrasings_right_%", "wrong_%", "refused_%", "failed_%", "unanswerable_refused_%",
        f"RS{RS_PENALTY}", f"RS{RS_PENALTY}_ci95", "passes"]
COST = ["config", "footprint_gb", "extra_memory_gb", "peak_gb_one_at_a_time", "latency_median_s", "latency_p90_s",
        "tokens_per_s", "batch_agreement", "batch_size_used"]
print("Accuracy and reliability (answerable items: test-suite accuracy; RS over all items):")
print(SUMMARY[SHOW].to_string(index=False))
print("\nMemory and speed:")
print(SUMMARY[COST].to_string(index=False))
print("\nTest-suite accuracy by question group (%):")
print(BY_GROUP.to_string())
print("\nBy phrasing (%):")
print(BY_PHRASING.to_string())
if len(BY_UNANSWERABLE):
    print("\nUnanswerable questions refused (%):")
    print(BY_UNANSWERABLE.to_string())
if len(PAIRS_DF):
    print("\nDifferences in test-suite accuracy (paired cluster bootstrap):")
    print(PAIRS_DF.to_string(index=False))

In [ ]:
# ── Cell 12: The decision (rule fixed 22 Oct), README.md and config.json ──
def as_markdown(df, index=False):
    try:
        return df.to_markdown(index=index)
    except ImportError:              # `tabulate` missing: a plain-text table
        return "```\n" + df.to_string(index=index) + "\n```"

S = SUMMARY.set_index("config")

def passed(cfg):
    return cfg in S.index and bool(S.loc[cfg, "passes"])

def said(cfg):
    if cfg not in S.index:
        return f"{cfg}: not run ({runs.get(cfg, {}).get('skipped', 'not in this run')})"
    r = S.loc[cfg]
    return (f"{cfg}: test-suite accuracy {r['ts_acc_%']}% {r['ts_acc_ci95']}, RS({RS_PENALTY}) "
            f"{r[f'RS{RS_PENALTY}']}, unanswerable refused {r['unanswerable_refused_%']}%: "
            f"{'PASS' if r['passes'] else 'FAIL'}")

def verdict(cfg, yes, no):
    return f"{cfg} was not run." if cfg not in S.index else (yes if passed(cfg) else no)

DECISION = [f"Pass = answerable test-suite accuracy of at least {PASS_TS_ACC:g}% and RS({RS_PENALTY}) above 0.", ""]
DECISION += [f"- {said(f'{m[0]} nf4')}" for m in MODELS] + [""]
DECISION.append("- Benchmark counting questions use the fixed query, not a model, whatever happens below.")
DECISION.append("- **Current system:** " + verdict("our-7B nf4", "our 7B writes the SQL; no second model.",
                                                   "our 7B in 4-bit does not meet the bar."))
DECISION.append("- **Final low-resource system (provisional):** " + verdict(
    "VL-3B nf4", "the base 3B passes, so the distilled 3B can probably write the SQL itself; re-run H2b on the "
                 "distilled 3B (L2) to confirm.", "the base 3B in 4-bit does not meet the bar."))
if not (passed("our-7B nf4") and passed("VL-3B nf4")):
    coders = sorted((c for c in ("Coder-1.5B nf4", "Coder-3B nf4") if passed(c)), key=lambda c: S.loc[c, "footprint_gb"])
    DECISION.append("- **Fallback:** " + (
        f"{coders[0]} passes and adds {S.loc[coders[0], 'footprint_gb']} GB; the team decides whether free-form "
        "questions are worth that memory." if coders else
        "no separate model passes either (or none was run): ship fixed queries only, no free-form questions."))
if passed("VL-3B nf4") and "our-7B nf4" in S.index and not passed("our-7B nf4"):
    DECISION.append("- The base 3B passes where our fine-tuned 7B fails, so our fine-tuning may have cost the 7B this "
                    "skill. To keep one model in the final system, consider adding a small set of question-SQL "
                    "pairs to L2's training.")
(OUT_DIR / "decision.md").write_text("# H2b decision (rule fixed 22 Oct, before any run)\n\n"
                                     + "\n".join(DECISION) + "\n", encoding="utf-8")

n_ans, n_un = int(ITEMS["answerable"].sum()), int((~ITEMS["answerable"]).sum())
agree = [r.get("batch_agreement") for r in runs.values() if r.get("batch_agreement") is not None]
(OUT_DIR / "README.md").write_text("\n".join([
    "# H2b: graded test of the SQL agent", "",
    f"{'SMOKE TEST. ' if SMOKE_TEST else ''}{len(ITEMS)} items: {n_ans} answerable ({len(ANSWERABLE)} questions, "
    f"{len(PHRASES)} phrasing(s), up to {IMAGES_PER_QUESTION} images each) and {n_un} unanswerable. Table: H2's "
    f"`buildings.sqlite` ({len(BUILDINGS):,} buildings in {len(IMAGES):,} images).", "",
    "## Decision", "", *DECISION, "",
    "## Accuracy and reliability", "", as_markdown(SUMMARY[SHOW]), "",
    "- `ts_acc_%`: test-suite accuracy on the answerable items: the final query matches the reference on all five "
    "versions of the table, with a 95% cluster-bootstrap interval.",
    "- `ex_real_acc_%`: the same, on the real table alone. The gap to `ts_acc_%` is answers right by chance.",
    "- `first_try_ts_acc_%`: before any retry. `unseen_acc_%`: question types no prompt example shows.",
    "- `all_phrasings_right_%`: instances answered correctly in all three phrasings.",
    "- `wrong_%`, `refused_%`, `failed_%`: the answerable items' outcomes (a query still failing after the retries "
    "counts as a refusal in RS).",
    f"- `RS{RS_PENALTY}`: reliability score over all items: +1 right answer or refusal of an unanswerable question, "
    f"0 refusal of an answerable one, -{RS_PENALTY} wrong answer or answer to an unanswerable question.", "",
    "## Memory and speed", "", as_markdown(SUMMARY[COST]), "",
    "- `extra_memory_gb`: what the model adds to the deployed system (a VLM the system already runs adds nothing).",
    "- Latency and peak memory: one question at a time, as deployed, on a T4.", "",
    "## By question group (test-suite accuracy, %)", "", as_markdown(BY_GROUP, index=True), "",
    "## By phrasing (%)", "", as_markdown(BY_PHRASING, index=True), "",
    *(["## Unanswerable questions refused (%)", "", as_markdown(BY_UNANSWERABLE, index=True), ""]
      if len(BY_UNANSWERABLE) else []),
    *(["## Differences between configurations (paired cluster bootstrap)", "", as_markdown(PAIRS_DF), ""]
      if len(PAIRS_DF) else []),
    "## Notes", "",
    "- The question bank, the prompt (`prompt.txt`) and the decision rule were fixed before any run.",
    "- Every model decodes greedily with no repetition penalty.",
    f"- Batched replies matched one-at-a-time replies on {min(agree):.0%} to {max(agree):.0%} of the check items; "
    f"a configuration below {AGREE_MIN:.0%} would have been decoded one at a time." if agree else
    "- No configuration ran.",
    f"- Versions of the table left out as ambiguous (ties, thresholds, midlines): {sum(left_out.values())}.",
    "- The base VL-3B stands in for the distilled 3B, which doesn't exist yet: re-run on it after L2.", "",
    "## References", "",
    "Zhong, Yu & Klein, EMNLP 2020 (test-suite accuracy); Lee et al. 2024, TrustSQL, and the EHRSQL 2024 shared "
    "task (reliability score); Chang et al., ICLR 2023, Dr.Spider (paraphrase robustness); Gao et al., VLDB 2024, "
    "DAIL-SQL, and Rajkumar et al. 2022 (prompt representation); Chen et al., ICLR 2024, Self-Debugging (retries "
    "with execution feedback); Hui et al. 2024, Qwen2.5-Coder.", "",
    "## Files", "",
    "`summary.csv`, `by_group.csv`, `by_phrasing.csv`, `by_unanswerable.csv`, `pairwise.csv`, `graded.csv` (one row "
    "per configuration and item), `items.csv`, `gold.json`, `answers.jsonl` (every reply and attempt), `runs.jsonl`, "
    "`prompt.txt`, `decision.md`, `config.json`",
]), encoding="utf-8")
config = {"task": "H2b", "smoke_test": SMOKE_TEST, "seed": SEED, "table": str(H2_DB), "items": len(ITEMS),
          "answerable_items": n_ans, "unanswerable_items": n_un, "questions": len(ANSWERABLE),
          "unanswerable_questions": len(UNANSWERABLE), "phrasings": PHRASES, "images_per_question": IMAGES_PER_QUESTION,
          "pool_images": POOL_IMAGES, "variants": VARIANTS, "versions_left_out": sum(left_out.values()),
          "models": MODELS, "precisions": PRECISIONS, "batch_size": BATCH_SIZE, "agree_min": AGREE_MIN,
          "max_new_tokens": MAX_NEW_TOKENS, "max_sql_retries": MAX_SQL_RETRIES, "pass_ts_acc": PASS_TS_ACC,
          "rs_penalty": RS_PENALTY, "bootstrap": BOOTSTRAP, "runs": runs, "versions": globals().get("VERSIONS"),
          "run_time_min": round((time.time() - t_run) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")
print("\n".join(DECISION))
print(f"\n✓ Wrote the outputs to {OUT_DIR}")